# MMR and Retrieval Evaluation Metrics

This is the final notebook in our Vector Embeddings & RAG section. We've built retrieval systems from simple vector search all the way through hybrid search, query expansion, RRF, HyDE, and parent document retrieval. Now it's time to answer the critical question: how do we know if any of this actually works?

Retrieval quality determines RAG quality. If you retrieve the wrong documents, even the best LLM can't help. This notebook covers:

1. **Maximal Marginal Relevance (MMR)**: Balances relevance with diversity to avoid redundant results
2. **Retrieval Metrics**: Precision, Recall, MRR, NDCG, MAP - how to measure if your retrieval actually works

Without metrics, you're flying blind. With MMR, you get diverse, non-redundant results.

In [ ]:
%pip install -q openai numpy

In [ ]:
import os
import numpy as np
from getpass import getpass
from dataclasses import dataclass, field
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_embeddings_batch(texts: list[str]) -> list[np.ndarray]:
    """Get embeddings for multiple texts efficiently."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [np.array(item.embedding) for item in response.data]

---

## Part 1: Maximal Marginal Relevance (MMR)

### The Redundancy Problem

Standard retrieval returns the top-k most similar documents. But similar documents are often... similar to each other. You get redundant information.

**Example query**: "What is machine learning?"

Standard top-3 might return:
1. "Machine learning is a subset of AI that enables systems to learn from data."
2. "ML is an AI technique where systems learn patterns from data."
3. "Machine learning allows computers to learn from data without explicit programming."

All three say essentially the same thing. We want diversity!

### The MMR Formula

$$\text{MMR} = \arg\max_{d_i \in R \setminus S} \left[ \lambda \cdot \text{sim}(q, d_i) - (1 - \lambda) \cdot \max_{d_j \in S} \text{sim}(d_i, d_j) \right]$$

Where:
- $q$ = query
- $R$ = candidate documents
- $S$ = already selected documents
- $\lambda$ = relevance vs diversity trade-off (0 to 1)

**In plain English:** MMR picks documents one at a time. For each pick, it asks two questions: "How relevant is this document to the query?" and "How different is it from documents I've already picked?" Lambda controls the balance -- high lambda (close to 1) means "just give me the most relevant results, even if they repeat each other," while low lambda (close to 0) means "give me diverse results even if some are slightly less relevant." The formula subtracts a penalty for similarity to already-selected documents, so each new pick is pushed toward covering new ground.

In [ ]:
# Sample documents about Python (intentionally includes similar docs)
documents = [
    # Similar: Python basics
    "Python is a high-level programming language known for its clear syntax and readability.",
    "Python is an interpreted language with simple, readable syntax that's easy to learn.",
    "The Python programming language emphasizes code readability and clean syntax.",
    
    # Similar: Data science
    "Python is widely used in data science with libraries like pandas and numpy.",
    "Data scientists prefer Python for its rich ecosystem including pandas, numpy, and scikit-learn.",
    
    # Similar: Web development
    "Django and Flask are popular Python web frameworks for building web applications.",
    "Python web development commonly uses frameworks like Django or Flask.",
    
    # Unique topics
    "Python's GIL (Global Interpreter Lock) can limit multi-threaded performance.",
    "List comprehensions in Python provide a concise way to create lists.",
    "Python 3.10 introduced structural pattern matching with the match statement."
]

# Embed all documents
print("Embedding documents...")
doc_embeddings = get_embeddings_batch(documents)
print(f"Embedded {len(documents)} documents.")

In [ ]:
def standard_retrieve(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Standard retrieval: return top-k by similarity."""
    query_embedding = get_embedding(query)
    
    # Score all documents
    scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    
    # Sort by score
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Test standard retrieval
query = "What is Python?"
print(f"Query: {query}\n")
print("=== Standard Retrieval (top-5) ===")
results = standard_retrieve(query, top_k=5)
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. [{score:.3f}] {doc}")

In [ ]:
def mmr_retrieve(
    query: str,
    top_k: int = 5,
    lambda_param: float = 0.5
) -> list[tuple[str, float, float]]:
    """
    MMR retrieval: balance relevance with diversity.
    
    Args:
        query: Search query
        top_k: Number of documents to return
        lambda_param: 0 = max diversity, 1 = max relevance
    
    Returns:
        List of (document, relevance_score, mmr_score) tuples
    """
    query_embedding = get_embedding(query)
    
    # Compute query-document similarities
    query_similarities = [
        cosine_similarity(query_embedding, doc_emb) 
        for doc_emb in doc_embeddings
    ]
    
    # Track selected documents
    selected_indices = []
    remaining_indices = list(range(len(documents)))
    results = []
    
    for _ in range(top_k):
        if not remaining_indices:
            break
        
        best_idx = None
        best_mmr = float('-inf')
        
        for idx in remaining_indices:
            # Relevance to query
            relevance = query_similarities[idx]
            
            # Max similarity to already selected documents
            if selected_indices:
                max_sim_to_selected = max(
                    cosine_similarity(doc_embeddings[idx], doc_embeddings[sel_idx])
                    for sel_idx in selected_indices
                )
            else:
                max_sim_to_selected = 0
            
            # MMR score
            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim_to_selected
            
            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx
        
        # Select the best document
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
        results.append((documents[best_idx], query_similarities[best_idx], best_mmr))
    
    return results

# Test MMR retrieval
print(f"Query: {query}\n")
print("=== MMR Retrieval (lambda=0.5, top-5) ===")
results = mmr_retrieve(query, top_k=5, lambda_param=0.5)
for i, (doc, rel_score, mmr_score) in enumerate(results, 1):
    print(f"{i}. [rel:{rel_score:.3f}, mmr:{mmr_score:.3f}] {doc}")

In [ ]:
# Compare different lambda values
print(f"Query: {query}\n")

for lambda_val in [1.0, 0.7, 0.5, 0.3, 0.0]:
    print(f"\n=== Lambda = {lambda_val} ===")
    if lambda_val == 1.0:
        print("(Pure relevance - same as standard retrieval)")
    elif lambda_val == 0.0:
        print("(Pure diversity - ignores relevance after first pick)")
    
    results = mmr_retrieve(query, top_k=3, lambda_param=lambda_val)
    for i, (doc, rel_score, mmr_score) in enumerate(results, 1):
        print(f"  {i}. {doc[:70]}...")

### MMR Lambda Guidelines

| Lambda | Behavior | Use Case |
|--------|----------|----------|
| 0.9-1.0 | Nearly pure relevance | When accuracy is critical |
| 0.7-0.8 | Slight diversity | General-purpose RAG |
| 0.5-0.6 | Balanced | Research, exploration |
| 0.3-0.4 | High diversity | Creative tasks, brainstorming |
| 0.0-0.2 | Nearly pure diversity | Sampling diverse examples |

---

## Part 2: Retrieval Evaluation Metrics

To improve retrieval, you need to measure it. Here are the key metrics:

| Metric | What It Measures | Range |
|--------|------------------|-------|
| Precision@K | % of top-K results that are relevant | 0-1 |
| Recall@K | % of relevant docs found in top-K | 0-1 |
| MRR | Rank of first relevant result | 0-1 |
| NDCG@K | Quality with position discount | 0-1 |
| MAP | Average precision across queries | 0-1 |

In [ ]:
# Create evaluation dataset with ground truth
@dataclass
class EvalQuery:
    """A query with known relevant documents."""
    query: str
    relevant_doc_indices: list[int]  # Indices of relevant documents
    relevance_grades: dict[int, int] = field(default_factory=dict)  # For graded relevance (NDCG)

# Define evaluation queries with ground truth
eval_queries = [
    EvalQuery(
        query="What is Python and how readable is it?",
        relevant_doc_indices=[0, 1, 2],  # The three "basics" documents
        relevance_grades={0: 3, 1: 3, 2: 3, 3: 1, 4: 1}  # 3=highly relevant, 1=somewhat
    ),
    EvalQuery(
        query="Python for data science and analytics",
        relevant_doc_indices=[3, 4],  # Data science documents
        relevance_grades={3: 3, 4: 3, 0: 1, 1: 1, 2: 1}
    ),
    EvalQuery(
        query="Building web apps with Python",
        relevant_doc_indices=[5, 6],  # Web development documents
        relevance_grades={5: 3, 6: 3}
    ),
    EvalQuery(
        query="Python performance and threading",
        relevant_doc_indices=[7],  # GIL document
        relevance_grades={7: 3}
    ),
    EvalQuery(
        query="Python syntax features and list comprehensions",
        relevant_doc_indices=[8, 9, 0, 1, 2],  # Syntax-related
        relevance_grades={8: 3, 9: 2, 0: 2, 1: 2, 2: 2}
    )
]

print(f"Created {len(eval_queries)} evaluation queries with ground truth.")

In [ ]:
def precision_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """
    Precision@K: What fraction of top-K results are relevant?
    
    Precision@K = (# relevant in top-K) / K
    """
    top_k = retrieved_indices[:k]
    relevant_in_top_k = sum(1 for idx in top_k if idx in relevant_indices)
    return relevant_in_top_k / k

def recall_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """
    Recall@K: What fraction of relevant docs are in top-K?
    
    Recall@K = (# relevant in top-K) / (# total relevant)
    """
    if not relevant_indices:
        return 0.0
    top_k = retrieved_indices[:k]
    relevant_in_top_k = sum(1 for idx in top_k if idx in relevant_indices)
    return relevant_in_top_k / len(relevant_indices)

# Test on a single query
test_query = eval_queries[0]
print(f"Query: {test_query.query}")
print(f"Relevant docs: {test_query.relevant_doc_indices}")

# Get retrieval results
results = standard_retrieve(test_query.query, top_k=10)
retrieved_indices = [documents.index(doc) for doc, _ in results]
print(f"Retrieved order: {retrieved_indices}\n")

for k in [1, 3, 5, 10]:
    p = precision_at_k(retrieved_indices, test_query.relevant_doc_indices, k)
    r = recall_at_k(retrieved_indices, test_query.relevant_doc_indices, k)
    print(f"K={k}: Precision={p:.2f}, Recall={r:.2f}")

### Understanding Precision@K and Recall@K

**Precision@K** = (number of relevant documents in your top-K results) / K

$$\text{Precision@K} = \frac{|\{\text{relevant docs}\} \cap \{\text{top-K retrieved}\}|}{K}$$

**In plain English:** Out of the K results you showed the user, how many were actually useful? If you return 5 results and 3 are relevant, your Precision@5 is 0.6 (60%). Think of it as "what percentage of my results are worth showing?"

---

**Recall@K** = (number of relevant documents in your top-K results) / (total number of relevant documents that exist)

$$\text{Recall@K} = \frac{|\{\text{relevant docs}\} \cap \{\text{top-K retrieved}\}|}{|\{\text{all relevant docs}\}|}$$

**In plain English:** Out of ALL the relevant documents that exist in your collection, how many did you manage to find in your top-K? If there are 4 relevant documents total and your top-5 results contain 2 of them, your Recall@5 is 0.5 (50%). Think of it as "what percentage of the good stuff did I actually find?"

There is a natural tension between precision and recall. Returning more results (higher K) tends to improve recall (you find more relevant docs) but often hurts precision (you also include more noise).

In [ ]:
def reciprocal_rank(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """
    Reciprocal Rank: 1 / (rank of first relevant result)
    
    If first relevant is at position 1: RR = 1.0
    If first relevant is at position 2: RR = 0.5
    If first relevant is at position 3: RR = 0.333
    If no relevant found: RR = 0.0
    """
    for i, idx in enumerate(retrieved_indices):
        if idx in relevant_indices:
            return 1.0 / (i + 1)
    return 0.0

def mean_reciprocal_rank(queries: list[EvalQuery], retrieval_fn) -> float:
    """
    MRR: Average reciprocal rank across queries.
    """
    rr_sum = 0.0
    for q in queries:
        results = retrieval_fn(q.query, top_k=10)
        retrieved_indices = [documents.index(doc) for doc, _ in results]
        rr_sum += reciprocal_rank(retrieved_indices, q.relevant_doc_indices)
    return rr_sum / len(queries)

# Calculate MRR
mrr = mean_reciprocal_rank(eval_queries, standard_retrieve)
print(f"Mean Reciprocal Rank (MRR): {mrr:.3f}")

# Show individual RRs
print("\nIndividual Reciprocal Ranks:")
for q in eval_queries:
    results = standard_retrieve(q.query, top_k=10)
    retrieved_indices = [documents.index(doc) for doc, _ in results]
    rr = reciprocal_rank(retrieved_indices, q.relevant_doc_indices)
    print(f"  RR={rr:.3f}: {q.query[:50]}...")

### Understanding MRR (Mean Reciprocal Rank)

**Reciprocal Rank** = 1 / (position of the first relevant result)

$$\text{RR} = \frac{1}{\text{rank of first relevant result}}$$

$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

**In plain English:** MRR answers the question: "How far down the list does the user have to look before finding something useful?" If the very first result is relevant, the reciprocal rank is 1/1 = 1.0 (perfect). If the first relevant result is at position 3, it is 1/3 = 0.33. MRR averages this across all your test queries. A high MRR means users usually find what they need near the top. MRR is especially useful for search engines and question-answering systems where the user mainly cares about the single best result.

In [ ]:
def dcg_at_k(retrieved_indices: list[int], relevance_grades: dict[int, int], k: int) -> float:
    """
    Discounted Cumulative Gain at K.
    
    DCG@K = sum_{i=1}^{K} (2^{rel_i} - 1) / log2(i + 1)
    
    Higher relevance grades contribute more, but position matters (logarithmic discount).
    """
    dcg = 0.0
    for i, idx in enumerate(retrieved_indices[:k]):
        rel = relevance_grades.get(idx, 0)
        dcg += (2**rel - 1) / np.log2(i + 2)  # i+2 because log2(1) = 0
    return dcg

def ndcg_at_k(retrieved_indices: list[int], relevance_grades: dict[int, int], k: int) -> float:
    """
    Normalized DCG at K.
    
    NDCG@K = DCG@K / IDCG@K
    
    Where IDCG is the ideal DCG (perfect ranking).
    Returns 0 if no relevant documents exist.
    """
    # Actual DCG
    dcg = dcg_at_k(retrieved_indices, relevance_grades, k)
    
    # Ideal DCG: sort by relevance grade (descending)
    ideal_order = sorted(relevance_grades.keys(), key=lambda x: relevance_grades[x], reverse=True)
    idcg = dcg_at_k(ideal_order, relevance_grades, k)
    
    if idcg == 0:
        return 0.0
    return dcg / idcg

# Test NDCG
test_query = eval_queries[0]
print(f"Query: {test_query.query}")
print(f"Relevance grades: {test_query.relevance_grades}")

results = standard_retrieve(test_query.query, top_k=10)
retrieved_indices = [documents.index(doc) for doc, _ in results]
print(f"Retrieved order: {retrieved_indices}\n")

for k in [1, 3, 5, 10]:
    dcg = dcg_at_k(retrieved_indices, test_query.relevance_grades, k)
    ndcg = ndcg_at_k(retrieved_indices, test_query.relevance_grades, k)
    print(f"K={k}: DCG={dcg:.3f}, NDCG={ndcg:.3f}")

### Understanding DCG and NDCG

**DCG (Discounted Cumulative Gain):**

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$$

**In plain English:** DCG rewards putting highly relevant documents at the top of the results list. Each document earns points based on how relevant it is (the numerator: $2^{\text{rel}} - 1$), but those points get discounted the further down the list the document appears (the denominator: $\log_2(i+1)$). A highly relevant document at position 1 contributes far more to the score than the same document at position 10. This reflects how real users behave -- they pay more attention to the first few results and rarely scroll deep into the list.

---

**NDCG (Normalized Discounted Cumulative Gain):**

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

**In plain English:** Raw DCG scores are hard to compare because they depend on how many relevant documents exist. NDCG fixes this by dividing your actual DCG by the "ideal" DCG (IDCG) -- the score you would get if your ranking were absolutely perfect. This gives you a number between 0 and 1. An NDCG of 1.0 means your ranking is ideal, 0.5 means you are roughly halfway there, and 0.0 means none of the relevant documents appeared in your top-K. NDCG is one of the most widely used retrieval metrics because it handles graded relevance (not just "relevant or not") and is comparable across different queries.

In [ ]:
def average_precision(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """
    Average Precision: Average of precision values at each relevant document.
    
    AP = (1/|R|) * sum_{k: d_k is relevant} Precision@k
    
    Rewards retrieving relevant documents early.
    """
    if not relevant_indices:
        return 0.0
    
    precision_sum = 0.0
    relevant_found = 0
    
    for i, idx in enumerate(retrieved_indices):
        if idx in relevant_indices:
            relevant_found += 1
            precision_at_i = relevant_found / (i + 1)
            precision_sum += precision_at_i
    
    return precision_sum / len(relevant_indices)

def mean_average_precision(queries: list[EvalQuery], retrieval_fn) -> float:
    """
    MAP: Average of Average Precision across queries.
    """
    ap_sum = 0.0
    for q in queries:
        results = retrieval_fn(q.query, top_k=len(documents))
        retrieved_indices = [documents.index(doc) for doc, _ in results]
        ap_sum += average_precision(retrieved_indices, q.relevant_doc_indices)
    return ap_sum / len(queries)

# Calculate MAP
map_score = mean_average_precision(eval_queries, standard_retrieve)
print(f"Mean Average Precision (MAP): {map_score:.3f}")

# Show individual APs
print("\nIndividual Average Precisions:")
for q in eval_queries:
    results = standard_retrieve(q.query, top_k=len(documents))
    retrieved_indices = [documents.index(doc) for doc, _ in results]
    ap = average_precision(retrieved_indices, q.relevant_doc_indices)
    print(f"  AP={ap:.3f}: {q.query[:50]}...")

### Understanding Average Precision and MAP

**Average Precision (AP):**

$$\text{AP} = \frac{1}{|\text{relevant docs}|} \sum_{k=1}^{n} \text{Precision@k} \cdot \mathbb{1}(d_k \text{ is relevant})$$

**In plain English:** Average Precision walks through your ranked result list from top to bottom. Every time it encounters a relevant document, it calculates the precision at that position and records it. Then it averages all those precision values. This means AP rewards you heavily for putting relevant documents early in the list. For example, if your only relevant document is at position 1, AP = 1.0. If it is at position 5, AP = 0.2. If you have two relevant documents at positions 1 and 3, AP = (1/1 + 2/3) / 2 = 0.83. AP captures the full shape of your ranking quality in a single number.

---

**MAP (Mean Average Precision):**

$$\text{MAP} = \frac{1}{|Q|} \sum_{q=1}^{|Q|} \text{AP}(q)$$

**In plain English:** MAP simply averages the AP scores across all your test queries. It gives you a single number that summarizes how well your retrieval system ranks results overall. MAP is one of the most popular metrics in information retrieval because it considers the entire ranking (not just the top-K) and penalizes both missing relevant documents and ranking them too low. A MAP of 0.9 means your system consistently ranks relevant documents near the top across many different queries.

---

## Part 3: Complete Evaluation Harness

Let's build a reusable evaluation harness that computes all metrics.

In [ ]:
@dataclass
class RetrievalMetrics:
    """Container for all retrieval metrics."""
    precision_at_1: float
    precision_at_3: float
    precision_at_5: float
    recall_at_1: float
    recall_at_3: float
    recall_at_5: float
    mrr: float
    ndcg_at_3: float
    ndcg_at_5: float
    map_score: float
    
    def __str__(self):
        return f"""
Retrieval Metrics:
  Precision:  @1={self.precision_at_1:.3f}  @3={self.precision_at_3:.3f}  @5={self.precision_at_5:.3f}
  Recall:     @1={self.recall_at_1:.3f}  @3={self.recall_at_3:.3f}  @5={self.recall_at_5:.3f}
  MRR:        {self.mrr:.3f}
  NDCG:       @3={self.ndcg_at_3:.3f}  @5={self.ndcg_at_5:.3f}
  MAP:        {self.map_score:.3f}
"""

def evaluate_retriever(
    queries: list[EvalQuery],
    retrieval_fn,
    k_values: list[int] = [1, 3, 5]
) -> RetrievalMetrics:
    """Compute all metrics for a retrieval function."""
    
    # Collect per-query metrics
    all_results = []
    for q in queries:
        results = retrieval_fn(q.query, top_k=max(k_values) + 5)
        retrieved_indices = [documents.index(doc) for doc, *_ in results]
        all_results.append((q, retrieved_indices))
    
    # Compute averages
    def avg(fn, k=None):
        if k:
            return np.mean([fn(r, q.relevant_doc_indices, k) for q, r in all_results])
        return np.mean([fn(r, q.relevant_doc_indices) for q, r in all_results])
    
    def avg_ndcg(k):
        return np.mean([ndcg_at_k(r, q.relevance_grades, k) for q, r in all_results])
    
    return RetrievalMetrics(
        precision_at_1=avg(precision_at_k, 1),
        precision_at_3=avg(precision_at_k, 3),
        precision_at_5=avg(precision_at_k, 5),
        recall_at_1=avg(recall_at_k, 1),
        recall_at_3=avg(recall_at_k, 3),
        recall_at_5=avg(recall_at_k, 5),
        mrr=avg(reciprocal_rank),
        ndcg_at_3=avg_ndcg(3),
        ndcg_at_5=avg_ndcg(5),
        map_score=np.mean([average_precision(r, q.relevant_doc_indices) for q, r in all_results])
    )

# Evaluate standard retrieval
print("=== Standard Retrieval ===")
metrics = evaluate_retriever(eval_queries, standard_retrieve)
print(metrics)

In [ ]:
# Create MMR retrieval functions with different lambdas
def mmr_retrieve_07(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.7)

def mmr_retrieve_05(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.5)

def mmr_retrieve_03(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.3)

# Compare all retrievers
retrievers = [
    ("Standard (λ=1.0)", standard_retrieve),
    ("MMR (λ=0.7)", mmr_retrieve_07),
    ("MMR (λ=0.5)", mmr_retrieve_05),
    ("MMR (λ=0.3)", mmr_retrieve_03),
]

print("Comparing Retrievers:")
print("=" * 70)

for name, fn in retrievers:
    metrics = evaluate_retriever(eval_queries, fn)
    print(f"\n{name}:")
    print(f"  P@3: {metrics.precision_at_3:.3f}  R@3: {metrics.recall_at_3:.3f}  MRR: {metrics.mrr:.3f}  MAP: {metrics.map_score:.3f}")

### Interpreting Metrics

| Metric | What It Tells You |
|--------|------------------|
| High Precision, Low Recall | Good at being accurate, misses some relevant docs |
| Low Precision, High Recall | Finds everything, but includes noise |
| Low MRR | First relevant result is buried |
| High NDCG | Highly relevant docs rank higher than moderately relevant |
| MAP | Overall ranking quality across the full result list |

**Key insight**: MMR often slightly reduces precision/recall (by diversifying) but can improve user experience by showing non-redundant results.

---

## Part 4: Advanced MMR Variations

### Adaptive Lambda

Use high lambda (more relevance) for specific queries, low lambda (more diversity) for broad queries.

In [ ]:
def adaptive_mmr_retrieve(
    query: str,
    top_k: int = 5,
    specificity_threshold: float = 0.7
) -> list[tuple[str, float, float]]:
    """
    Adaptive MMR: Adjust lambda based on query specificity.
    
    Specific queries (high max similarity to any doc) -> high lambda
    Broad queries (moderate similarity to many docs) -> low lambda
    """
    query_embedding = get_embedding(query)
    
    # Compute max similarity to any document
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    max_sim = max(similarities)
    std_sim = np.std(similarities)
    
    # High max + low std = specific query (use high lambda)
    # Moderate max + high std = broad query (use low lambda)
    if max_sim > specificity_threshold and std_sim > 0.05:
        lambda_param = 0.8  # Specific query, prioritize relevance
    elif max_sim < specificity_threshold - 0.1:
        lambda_param = 0.4  # Vague query, prioritize diversity
    else:
        lambda_param = 0.6  # Balanced
    
    print(f"  [Adaptive: max_sim={max_sim:.2f}, std={std_sim:.2f} -> λ={lambda_param}]")
    return mmr_retrieve(query, top_k, lambda_param)

# Test adaptive MMR
test_queries = [
    "What is Python?",  # Broad
    "Python GIL threading performance",  # Specific
    "Python web frameworks Django Flask",  # Moderately specific
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = adaptive_mmr_retrieve(q, top_k=3)
    for i, (doc, rel, mmr) in enumerate(results, 1):
        print(f"  {i}. {doc[:60]}...")

---

## Exercises

### Exercise 1: MMR with Weights

Modify MMR to consider document importance weights (e.g., recency, popularity).

In [ ]:
def weighted_mmr_retrieve(
    query: str,
    doc_weights: list[float],  # Importance weight per document
    top_k: int = 5,
    lambda_param: float = 0.5,
    weight_factor: float = 0.2  # How much to factor in weights
) -> list[tuple[str, float, float]]:
    """MMR with document importance weights."""
    # Your implementation here
    pass

### Exercise 2: Custom Metric

Implement a metric that penalizes redundancy in retrieved results.

In [ ]:
def diversity_penalized_precision(
    retrieved_indices: list[int],
    relevant_indices: list[int],
    k: int,
    redundancy_penalty: float = 0.1
) -> float:
    """
    Precision@K with a penalty for redundant (similar) retrieved documents.
    """
    # Your implementation here
    pass

### Exercise 3: A/B Test Simulation

Simulate an A/B test between two retrieval strategies and determine statistical significance.

In [ ]:
def ab_test_retrievers(
    queries: list[EvalQuery],
    retriever_a,
    retriever_b,
    metric_fn,  # Function to compute metric
    n_bootstrap: int = 1000
) -> dict:
    """
    Compare two retrievers and compute confidence interval.
    """
    # Your implementation here
    pass

---

## Summary

### MMR
- Balances relevance with diversity to avoid redundant results
- Lambda parameter controls the trade-off (higher = more relevance)
- Use when showing multiple results to users (search, recommendations, RAG context)

### Retrieval Metrics

| Metric | Best For |
|--------|----------|
| Precision@K | "Are my top results good?" |
| Recall@K | "Am I finding everything relevant?" |
| MRR | "How quickly do users find what they need?" |
| NDCG | "Are better results ranked higher?" |
| MAP | "How good is my overall ranking?" |

**Key insight**: You can't improve what you don't measure. Build evaluation sets, compute metrics, iterate.

### Wrapping Up the Vector Embeddings & RAG Section

This concludes our Vector Embeddings & RAG section. We started with embeddings and simple vector search, built up through BM25, hybrid search, query expansion, RAG-Fusion with RRF, HyDE, and parent document retrieval. Now with MMR and evaluation metrics, you have the tools to not only build retrieval systems but measure and improve them.